# Wake Word Quickstart — Kaggle / Colab

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/OpenVoiceOS/ngi0-notebooks/blob/dev/notebooks/kaggle_quickstart_ww.ipynb) [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenVoiceOS/ngi0-notebooks/blob/dev/notebooks/kaggle_quickstart_ww.ipynb)

## Start here

This notebook makes a wake word model: a small file that lets OpenVoiceOS wake up when it hears your phrase. It takes about one to one and a half hours. It needs a GPU and the internet: on Kaggle, open the Settings panel, set Accelerator to a GPU and switch Internet on. The only thing to change is `WAKE_WORD` in the first code cell, if you want a phrase other than "hey mycroft". Then press **Run All** and read "What you got" at the bottom.

<details>
<summary>Already made voices with <code>synthetic_ww_dataset</code>?</summary>

Give their folders to `DATASET_DIRS` and this notebook trains on those voices: Step 3 reads them from the folders instead of generating them. On Kaggle, attach each finished run with Add Input, Notebook output, and set `DATASET_DIRS = "/kaggle/input"`. The folders must be for one phrase and come from runs with different `RUN_ID`s; the phrase is taken from them.

</details>

In [ ]:
WAKE_WORD = "hey mycroft"   # the phrase that wakes your assistant; any phrase works
LANGUAGE = "English"        # the language of the phrase
DATASET_DIRS = ""           # leave empty, or name folders made by synthetic_ww_dataset, such as "/kaggle/input"
SMOKE = False               # True: a five-minute trial that needs no GPU; its model is not good enough to use

# Sizes. Leave these alone.
N_SAMPLES = 16              # SMOKE only: how many times the phrase is spoken
EPOCHS = 40                 # how many times training goes over the examples

### Advanced settings

Everything else the notebook can be told. Skip this cell: the values below work as they are.

In [ ]:
import os

SEED = 42                                       # the same seed gives the same choices of clips
WORK_DIR = "/tmp/wakehubert_work" if os.path.isdir("/kaggle") else "./wakehubert_work"   # downloads; not part of the result
OUTPUT_DIR = "./wakehubert_output"              # the model file and the OVOS settings go here
SHOW_DETAILS = False                            # True: also print the tables behind every step

# How many clips
N_DESIGN_TRAIN, N_DESIGN_TEST = 1750, 230      # the phrase in an invented voice: for training, for the test
N_CLONE_TRAIN, N_CLONE_TEST = 210, 70          # the phrase in a copied voice, each from a different real speaker
MSWC_TEST_PERCENT = 20                          # share of the real speakers kept for the test; synthetic_ww_dataset uses the same rule
N_SPEECH_TRAIN, N_SPEECH_TEST, N_SPEECH_STREAM = 800, 100, 1200   # single spoken words that are not the phrase
N_SENTENCE_WINDOWS = 1200                       # 1.5 s pieces cut from read-aloud sentences
N_SOUND_TEST = 120                              # sound clips kept for the short test
N_SILENCE_TRAIN, N_SILENCE_TEST = 300, 40       # silence and faint hiss

# The long recordings (streams), in seconds
STREAM_SECONDS = 7200                           # the calibration stream and the test stream, each
SOUND_SHARE = 0.4                               # share of those two that is sound and music rather than speech
MINE_SOUND_SECONDS, MINE_SPEECH_SECONDS = 7200, 7500   # the training stream

# Training
LR, BATCH_SIZE = 2e-3, 64
MAX_MINING_ROUNDS, STOP_BELOW = 5, 5            # retrain until fewer than 5 training-stream windows score 0.5 or more, at most 5 times
MINE_FROM, MAX_HARD_PER_ROUND, MAX_HARD_TOTAL, MAX_RANDOM_STREAM = 0.1, 6000, 20000, 16000   # training-stream windows scoring above MINE_FROM become negatives: at most this many per round and in all, plus this many random ones
SMOKE_MIN_EPOCHS, SMOKE_LR = 30, 5e-3           # SMOKE trains at least this many epochs, at this learning rate

# Threshold: the lowest of these with at most CAL_MAX_FA_PER_HOUR false wake-ups per hour on the calibration stream
CANDIDATES = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.97, 0.98, 0.99, 0.995, 0.998, 0.999]
CAL_MAX_FA_PER_HOUR = 2.0

## Step 1 — Install and check the machine

Installs what the notebook needs and checks for the internet and a GPU. If a red box appears, read its last line: it says what to click.

In [ ]:
#@title Code for step 1: checks the DATASET_DIRS folders, the internet, the GPU and the disk
import io, json, math, random, shutil, sys, time, urllib.error, urllib.request, warnings, zlib
from collections import Counter
from pathlib import Path

def detail(*a, **k):
    # SHOW_DETAILS: a line of the tables and traces behind a step, with the clock time
    if SHOW_DETAILS:
        print(time.strftime("%H:%M:%S"), *a, **k)
if not SHOW_DETAILS:
    os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # no download bars; the step lines say what happened
_STEPS, _last, _done = {}, [time.time()], [time.time()]
def tick(name):
    # records the seconds since the previous tick, for the summary at the end
    now = time.time()
    _STEPS[name], _last[0] = now - _last[0], now
def done(name, message):
    # the step's one line, with the clock time and the seconds the step took
    tick(name)
    print(f"{message} [{time.strftime('%H:%M:%S')}, {_last[0] - _done[0]:.0f} s]")
    _done[0] = _last[0]

DONOR_SPLIT = f"MLCommons/ml_spoken_words speaker_id, crc32 % 100 < {MSWC_TEST_PERCENT} is test"
ATTACH = 'On Kaggle: Add Input, Notebook output, pick your synthetic_ww_dataset runs, then set DATASET_DIRS = "/kaggle/input".'

def find_datasets(dirs):
    # [(folder, its dataset.json)] for every folder under `dirs` that synthetic_ww_dataset wrote
    datasets = []
    for root in ([p.strip() for p in dirs.replace("\n", ",").split(",") if p.strip()] if isinstance(dirs, str) else dirs):
        root = os.path.abspath(os.path.expanduser(root))
        assert os.path.isdir(root), f"DATASET_DIRS names {root}, which is not a folder. {ATTACH}"
        found, seen, visited = [], [], set()
        for d, subdirs, files in os.walk(root, followlinks=True):   # Kaggle mounts attached inputs a few levels down, sometimes as links
            real = os.path.realpath(d)
            if real in visited:                     # a link to a folder already searched, or a link loop
                subdirs[:] = []
                continue
            visited.add(real)
            seen.append(d)
            if "dataset.json" in files and "metadata.csv" in files:
                found.append(real)
                subdirs[:] = []                     # a dataset's own audio folders are not searched
            elif d[len(root):].count(os.sep) >= 6:
                subdirs[:] = []
        assert found, (f"No folder written by synthetic_ww_dataset (one holding dataset.json and metadata.csv) was found under {root}. {ATTACH} "
                       f"Folders searched: {', '.join(seen[:12])}{' ...' if len(seen) > 12 else ''}")
        for d in sorted(found):
            if d not in (x for x, _ in datasets):
                datasets.append((d, json.load(open(os.path.join(d, "dataset.json"), encoding="utf-8"))))
    if not datasets:
        return datasets
    norm = lambda s: " ".join(str(s).lower().split())
    listing = "; ".join(f"{d}: {m.get('wake_word')!r} ({m.get('language')}), run {m.get('run_id')}" for d, m in datasets)
    if any(m.get("format") != "ovos-synthetic-ww-dataset" for _, m in datasets):
        raise ValueError(f"A folder in DATASET_DIRS was not written by synthetic_ww_dataset: {listing}")
    if len({norm(m["wake_word"]) for _, m in datasets}) > 1 or len({m["language"] for _, m in datasets}) > 1:
        raise ValueError(f"The folders in DATASET_DIRS are for different phrases; give it folders made for one phrase. Found: {listing}")
    twice = [r for r, n in Counter(m["run_id"] for _, m in datasets).items() if n > 1]
    if twice:
        raise ValueError(f"Several folders were made with RUN_ID {twice}, so their voices are the same voices. "
                         f"Generate each folder with its own RUN_ID. Found: {listing}")
    old = [d for d, m in datasets if m.get("donor_split") != DONOR_SPLIT]
    if old:
        raise ValueError(f"{', '.join(old)} was made by an older synthetic_ww_dataset that split its copied voices by another rule, so its test "
                         "voices could be heard in training here. Make it again with the current synthetic_ww_dataset.")
    if not SMOKE and any(m.get("smoke") for _, m in datasets):
        raise ValueError("A folder in DATASET_DIRS was made with SMOKE = True (espeak-ng, the CI path only); it is not training data.")
    return datasets

def check_internet():
    try:
        urllib.request.urlopen("https://huggingface.co", timeout=20)
    except urllib.error.HTTPError:
        pass                                        # the site answered, so the internet is on
    except OSError as exc:
        raise RuntimeError("No internet. Kaggle: open the Settings panel, switch Internet on, then press Run All again.") from exc

def check_machine():
    # "cuda" when a GPU really computes, else "cpu"; the real run stops without a GPU
    import torch
    from importlib.metadata import version
    device = "cpu"
    if torch.cuda.is_available():
        try:
            (torch.ones(8, device="cuda") * 2).sum().item()
            device = "cuda"
        except Exception as exc:
            print(f"A GPU is listed but cannot run a calculation ({exc}); using the CPU.")
    if not SMOKE:
        assert device == "cuda", ("The real run needs a GPU. Kaggle: open the Settings panel, set Accelerator to a GPU, then press Run All again. "
                                  "Colab: Runtime, Change runtime type, GPU. Or set SMOKE = True in the first code cell for a small trial.")
    elif not DATASETS:
        assert shutil.which("espeak-ng"), "SMOKE speaks the phrase with espeak-ng, which is not installed (Linux: apt-get install espeak-ng)."
    os.makedirs(WORK_DIR, exist_ok=True)
    free_gb, need_gb = shutil.disk_usage(WORK_DIR).free / 1e9, 2 if SMOKE else 8
    assert free_gb > need_gb, (f"Only {free_gb:.1f} GB of disk is free; this run needs about {need_gb} GB. Kaggle: stop the session and start it "
                               "again. Colab: Runtime, Disconnect and delete runtime, then open the notebook again.")
    detail(f"Python {sys.version.split()[0]} | torch {torch.__version__} | device {device}" + (f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else "")
           + f" | ovos-ww-plugin-wakeforge {version('ovos-ww-plugin-wakeforge')}" + ("" if OMNIVOICE == "" else f" | omnivoice {version('omnivoice')}")
           + f" | {THREADS} CPU threads | {free_gb:.1f} GB of disk free")
    return device

if SMOKE:                                           # the small trial: a handful of robotic voices and very short streams
    N_DESIGN_TRAIN, N_DESIGN_TEST = N_SAMPLES, 4
    N_CLONE_TRAIN, N_CLONE_TEST = 2, 1              # picked and downloaded, not spoken: SMOKE speaks with espeak-ng
    N_SPEECH_TRAIN, N_SPEECH_TEST, N_SPEECH_STREAM = 2 * N_SAMPLES, 4, 20
    N_SENTENCE_WINDOWS, N_SOUND_TEST = 0, 6
    N_SILENCE_TRAIN, N_SILENCE_TEST = 2 * N_SAMPLES, 4
    STREAM_SECONDS = MINE_SOUND_SECONDS = MINE_SPEECH_SECONDS = 0
    MAX_MINING_ROUNDS, EPOCHS, LR = 1, max(EPOCHS, SMOKE_MIN_EPOCHS), SMOKE_LR
DATASETS = find_datasets(DATASET_DIRS)
if DATASETS:
    WAKE_WORD, LANGUAGE = DATASETS[0][1]["wake_word"], DATASETS[0][1]["language"]
    N_CLONE_TRAIN = N_CLONE_TEST = 0                # the folders bring their own voices
    print(f"Using the voices in {len(DATASETS)} folder(s) from DATASET_DIRS, made for the phrase {WAKE_WORD!r}. No voices are generated here.")
    for d, m in DATASETS:
        detail(f"   {d}: run {m['run_id']}, {m['counts']['positives']['train']} train and {m['counts']['positives']['test']} test clips")
WORD_SLUG = WAKE_WORD.strip().lower().replace(" ", "_").replace("-", "_")
THREADS = min(4, os.cpu_count() or 1)
OMNIVOICE = "" if SMOKE or DATASETS else '"omnivoice>=0.2.1"'   # the speech generator; only the run that generates voices needs it

In [ ]:
check_internet()
%pip install --quiet --pre "ovos-ww-plugin-wakeforge>=0.17.0a1" "onnx>=1.16" "soundfile>=0.12" "scipy>=1.11" "huggingface_hub>=0.30" "pyarrow>=15" {OMNIVOICE}
DEVICE = check_machine()
done("install and setup", f"Step 1 done: ready to make a wake word for {WAKE_WORD!r} on " + ("a GPU." if DEVICE == "cuda" else "the CPU (small trial only)."))

## Step 2 — Download practice sounds

Downloads everyday speech, noises and music. The model learns that these must not wake it.

<details>
<summary>Where the sounds come from</summary>

Single spoken words, noises and music come from the [`TigreGotico/notwakeword-datasets`](https://huggingface.co/collections/TigreGotico/notwakeword-datasets) collection and from AudioSet. Read-aloud sentences come from LibriSpeech, about 700 MB. Room echoes come from `davidscripka/MIT_environmental_impulse_responses`. The real voices to copy in Step 3 are single recorded words of [`MLCommons/ml_spoken_words`](https://huggingface.co/datasets/MLCommons/ml_spoken_words) in the language of your phrase. Part of every source is kept aside and used only in the tests; pieces cut from one recording stay on the same side.

</details>

In [ ]:
#@title Code for step 2: reads audio and downloads the speech, the sounds, the room echoes and the voices to copy
import re, tarfile
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import onnxruntime as ort
import pyarrow.parquet as pq
import soundfile as sf
import torch
from scipy.signal import fftconvolve, resample_poly
from huggingface_hub import HfApi, HfFileSystem, hf_hub_download, snapshot_download

FEATURIZER_NAME, FEATURIZER_REPO = "wakehubert-int8", "TigreGotico/wakehubert-tiny"
FEATURIZER_REVISION = "343b26ea8881b5a01447a1659555941b25220d22"
SR, WINDOW, HOP = 16000, 24000, 1280       # 16 kHz audio; a window is 1.5 s; the plugin scores one every 80 ms
N_FRAMES, FEATURE_DIM = 75, 128             # what the featurizer gives for one window
AUDIO_EXT = (".wav", ".mp3", ".flac", ".ogg")
COLLECTION = "TigreGotico/notwakeword-datasets"
SPEECH_REPO = "TigreGotico/not-wake-words-speech-en"
RIR_REPO = "davidscripka/MIT_environmental_impulse_responses"
# Sound and music sets of the collection, read as a few Parquet shards each. The number is the most clips taken from a set.
SOUND_SETS = ({"TigreGotico/public_domain_sounds_3secs": 8, "TigreGotico/FMA_3secs": 8, "TigreGotico/NAR": 8, "TigreGotico/ambient_noises": 8} if SMOKE else
              {"TigreGotico/public_domain_sounds_3secs": 2500, "TigreGotico/ambient_noises": 1500, "TigreGotico/ESC-50": 1500,
               "TigreGotico/FMA_3secs": 503, "TigreGotico/building_106_kitchen_3secs": 800, "TigreGotico/NAR": 600})
AUDIOSET = "agkphysics/AudioSet"            # balanced set; 10 s clips, a few of its 700 MB shards
AUDIOSET_TRAIN_SHARDS, AUDIOSET_TEST_SHARD, AUDIOSET_STREAM_SHARD, AUDIOSET_PER_SHARD = [3, 17, 31], 4, 20, 600
MSWC_REPO = "MLCommons/ml_spoken_words"
# The Multilingual Spoken Words config of each language, by OmniVoice's name for it (a few common English names besides).
# LANGUAGE may also be a config's code itself, the part before "_opus" (such as fy-NL or rm-vallader).
MSWC_CODES = {"standard arabic": "ar", "arabic": "ar", "assamese": "as", "basque": "eu", "breton": "br", "catalan": "ca",
              "chinese": "zh-CN", "chuvash": "cv", "czech": "cs", "dhivehi": "dv", "dutch": "nl", "english": "en",
              "esperanto": "eo", "estonian": "et", "french": "fr", "georgian": "ka", "german": "de", "greek": "el",
              "guarani": "gn", "hakha chin": "cnh", "hausa": "ha", "indonesian": "id", "interlingua": "ia",
              "interlingua (international auxiliary language association)": "ia", "irish": "ga-IE", "italian": "it",
              "kinyarwanda": "rw", "kirghiz": "ky", "kyrgyz": "ky", "latvian": "lv", "lithuanian": "lt", "maltese": "mt",
              "mongolian": "mn", "odia": "or", "oriya": "or", "persian": "fa", "polish": "pl", "portuguese": "pt",
              "romanian": "ro", "romansh": "rm-sursilv", "russian": "ru", "sakha": "sah", "yakut": "sah", "slovak": "sk",
              "slovenian": "sl", "spanish": "es", "swedish": "sv-SE", "tamil": "ta", "tatar": "tt", "turkish": "tr",
              "ukrainian": "uk", "vietnamese": "vi", "welsh": "cy", "western frisian": "fy-NL", "frisian": "fy-NL"}
MSWC_CODES.update({c.lower(): c for c in [*MSWC_CODES.values(), "rm-vallader"]})

rng = random.Random(SEED)
api = HfApi()
opts = ort.SessionOptions()
opts.intra_op_num_threads, opts.inter_op_num_threads = THREADS, 1

def load_featurizer():
    # the frozen WakeHuBERT model, at a fixed revision; the plugin runs the same file
    path = hf_hub_download(FEATURIZER_REPO, "wakehubert_int8.onnx", revision=FEATURIZER_REVISION)
    sess = ort.InferenceSession(path, opts, providers=["CPUExecutionProvider"])
    assert sess.run(None, {sess.get_inputs()[0].name: np.zeros((1, WINDOW), np.float32)})[0].shape[1:] == (N_FRAMES, FEATURE_DIM)
    return sess

def featurize(windows):
    # windows: [n, 24000] float32 -> [n, 75, 128]. Each window is featurized alone, as the plugin does.
    name = feat_sess.get_inputs()[0].name
    return np.concatenate([feat_sess.run(None, {name: np.ascontiguousarray(windows[i:i + 32], np.float32)})[0]
                           for i in range(0, len(windows), 32)])

def to16k(x, sr):
    if sr == SR:
        return x
    g = math.gcd(SR, sr)
    return resample_poly(x, SR // g, sr // g).astype(np.float32)

def load16k(src):
    # a file path or the bytes of a file -> mono 16 kHz audio
    x, sr = sf.read(io.BytesIO(src) if isinstance(src, bytes) else src, dtype="float32", always_2d=True)
    return to16k(x.mean(axis=1), sr)

def trim(x, db=-40.0):
    peak = np.abs(x).max()
    if peak == 0:
        return x
    idx = np.nonzero(np.abs(x) > peak * 10 ** (db / 20))[0]
    return x[idx[0]: idx[-1] + 1]

def load_set(repo, n, folder):
    # n random audio files of a dataset: [(file name, audio)]
    files = sorted(f for f in api.list_repo_files(repo, repo_type="dataset") if f.lower().endswith(AUDIO_EXT))
    names = [f for f in rng.sample(files, min(n, len(files))) if not any(c in f for c in "[]*?")]
    snapshot_download(repo, repo_type="dataset", allow_patterns=names, local_dir=Path(WORK_DIR) / folder, max_workers=8)
    clips = []
    for f in (Path(WORK_DIR) / folder / n for n in names):
        try:
            clips.append((f.name, load16k(f)))
        except Exception as exc:                      # an unreadable file is skipped, not fatal
            detail(f"skipped {f.name}: {exc}")
    return clips

def parquet_clips(repo, filename, cap, tag, revision=None):
    # up to `cap` clips spread evenly over one Parquet shard
    pf = pq.ParquetFile(hf_hub_download(repo, filename, repo_type="dataset", revision=revision))
    col = next(f.name for f in pf.schema_arrow if "bytes" in str(f.type))
    stride, out, k = max(1, pf.metadata.num_rows // cap), [], 0
    for batch in pf.iter_batches(batch_size=64, columns=[col]):
        for item in batch.column(0).to_pylist():
            if k % stride == 0 and len(out) < cap and item.get("bytes"):
                try:
                    x = load16k(item["bytes"])
                except Exception:
                    x = None
                if x is not None and len(x) > 0.1 * SR and np.abs(x).max() > 1e-4:
                    out.append((f"{tag}/{Path(item.get('path') or '').name or filename}#{k}", x))
            k += 1
    return out

def recording(name):
    # the recording a clip was cut from: ESC-50 names are fold-clip-take-class, and two sets number the 3 s pieces of a recording
    tag, base = name.split("/", 1)
    base = base.split("#")[0]
    if tag == "ESC-50":
        return f"{tag}/" + base.split("-")[1]      # the Freesound clip id; a few appear in two folds
    if tag in ("public_domain_sounds_3secs", "building_106_kitchen_3secs"):
        return f"{tag}/" + re.sub(r"\d{3}\.wav$", "", base)
    return f"{tag}/{base}"

def split4(clips, r):
    # train / short test / calibration stream / test stream, by source recording: pieces of one recording stay together
    groups = {}
    for c in clips:
        groups.setdefault(recording(c[0]), []).append(c)
    keys = list(groups)
    r.shuffle(keys)
    n = len(clips)
    sizes = [max(1, round(0.04 * n)), max(1, round(0.13 * n)), max(1, round(0.13 * n))]
    parts = [[], [], [], []]
    for k in keys:                                   # a recording goes to the first held-out part with room for half of it
        i = next((j for j in range(3) if sizes[j] - len(parts[j]) >= len(groups[k]) / 2), 3)
        parts[i] += groups[k]
    return parts[3], parts[0], parts[1], parts[2]

def download_spoken_words():
    # single words, kept apart by use: training, the short test, the calibration stream, the test stream
    clips = [(n, trim(x)) for n, x in load_set(SPEECH_REPO, N_SPEECH_TRAIN + N_SPEECH_TEST + N_SPEECH_STREAM, "speech")]
    clips = [(n, x) for n, x in clips if len(x) > 0.1 * SR]
    detail(f"   {SPEECH_REPO}: {len(clips)} words")
    a, b = N_SPEECH_TRAIN, N_SPEECH_TRAIN + N_SPEECH_TEST
    stream = clips[b:b + N_SPEECH_STREAM]
    return {"train": clips[:a], "test": clips[a:b], "cal": stream[:len(stream) // 2], "tst": stream[len(stream) // 2:]}

def download_sounds():
    # noises and music, kept apart the same way
    sounds = {"train": [], "test": [], "cal": [], "tst": []}
    for repo, cap in SOUND_SETS.items():
        shards = sorted(f for f in api.list_repo_files(repo, repo_type="dataset", revision="refs/convert/parquet") if f.endswith(".parquet"))
        clips = []
        for fn in shards:
            clips += parquet_clips(repo, fn, -(-cap // len(shards)), repo.split("/")[1], revision="refs/convert/parquet")
        for key, part in zip(sounds, split4(clips[:cap], random.Random(f"{SEED}:{repo}"))):
            sounds[key] += part
        detail(f"   {repo}: {len(clips[:cap])} clips from {len({recording(n) for n, _ in clips[:cap]})} recordings")
    if not SMOKE:
        files = sorted(f for f in api.list_repo_files(AUDIOSET, repo_type="dataset") if f.endswith(".parquet"))
        bal, ev = [f for f in files if f.startswith("data/bal_train/")], [f for f in files if f.startswith("data/eval/")]
        for i in AUDIOSET_TRAIN_SHARDS:
            sounds["train"] += parquet_clips(AUDIOSET, bal[i], AUDIOSET_PER_SHARD, "AudioSet")
            detail(f"   AudioSet training shard {i} read")
        a = parquet_clips(AUDIOSET, ev[AUDIOSET_TEST_SHARD], AUDIOSET_PER_SHARD, "AudioSet")
        sounds["test"] += a[:N_SOUND_TEST]; sounds["cal"] += a[N_SOUND_TEST:]
        sounds["tst"] += parquet_clips(AUDIOSET, ev[AUDIOSET_STREAM_SHARD], AUDIOSET_PER_SHARD, "AudioSet")
    seen = {}
    for key, part in sounds.items():
        for rec in {recording(n) for n, _ in part}:
            assert seen.setdefault(rec, key) == key, f"pieces of the recording {rec} are in both the {seen[rec]} and the {key} sounds"
    for key, part in sounds.items():
        detail(f"   sounds, {key}: " + ", ".join(f"{k} {v}" for k, v in sorted(Counter(n.split("/")[0] for n, _ in part).items())))
    return sounds

def mswc_side(speaker):
    # training or test side of a donor, from the speaker id alone: the same in every run and in both wake-word notebooks
    return "test" if zlib.crc32(speaker.encode()) % 100 < MSWC_TEST_PERCENT else "train"

def find_voices_to_copy():
    # {side: [donor]}: one recorded word per real speaker, read a row group (100 recordings) at a time, never a whole file
    need, r = {"train": N_CLONE_TRAIN, "test": N_CLONE_TEST}, random.Random(f"{SEED}:donors")
    donors, stats = {"train": [], "test": []}, Counter()
    if not sum(need.values()):
        return donors
    code = MSWC_CODES.get(LANGUAGE.strip().lower())
    if code is None:
        raise ValueError(f"{LANGUAGE!r} is not in this notebook's table of Multilingual Spoken Words languages (MSWC_CODES). "
                         f"If {MSWC_REPO} has the language, set LANGUAGE to its config's code, the part before \"_opus\" "
                         "(such as fy-NL). Otherwise set N_CLONE_TRAIN, N_CLONE_TEST = 0, 0 in the Advanced settings cell to use invented voices only.")
    shards = sorted(f for f in api.list_repo_files(MSWC_REPO, repo_type="dataset", revision="refs/convert/parquet")
                    if f.startswith(f"{code}_opus/") and f.endswith(".parquet"))
    r.shuffle(shards)
    hfs = HfFileSystem()
    per_file = min(16, max(2, sum(need.values()) // 18))         # row groups read from one file
    at_once = min(4, max(1, sum(need.values()) // 70))          # files read in parallel

    def read(shard, seed):
        with hfs.open(f"datasets/{MSWC_REPO}@refs%2Fconvert%2Fparquet/{shard}", block_size=1 << 18, cache_type="bytes") as f:
            pf = pq.ParquetFile(f)
            groups = random.Random(seed).sample(range(pf.num_row_groups), min(per_file, pf.num_row_groups))
            tables = [pf.read_row_group(g, columns=["is_valid", "speaker_id", "keyword", "audio"]).to_pylist() for g in groups]
            return shard, groups, tables, f.cache.total_requested_bytes

    speakers = set()
    with ThreadPoolExecutor(at_once) as ex:
        for start in range(0, len(shards), at_once):
            if all(len(donors[s]) >= need[s] for s in need):
                break
            batch = shards[start:start + at_once]
            for shard, groups, tables, nbytes in ex.map(read, batch, [r.random() for _ in batch]):
                stats["bytes"] += nbytes
                stats["files"] += 1
                for g, rows in zip(groups, tables):
                    stats["rows read"] += len(rows)
                    r.shuffle(rows)
                    taken = Counter()                       # a row group holds one word, so only a few speakers are taken from each
                    for row in rows:
                        spk, s = row["speaker_id"], mswc_side(row["speaker_id"])
                        if (len(donors[s]) >= need[s] or taken[s] >= (5 if s == "train" else 3) or spk in speakers
                                or not row["is_valid"] or not (row["keyword"] or "").strip()):
                            continue
                        try:
                            x = load16k(row["audio"]["bytes"])
                        except Exception:                   # an undecodable recording is skipped
                            continue
                        if not len(x) or np.abs(x).max() < 0.005 or len(trim(x)) < 0.3 * SR:   # silent or too short
                            continue
                        donors[s].append({"speaker": spk, "keyword": row["keyword"].strip(), "audio": x,
                                          "recording": f"{code}_opus/{row['audio']['path']}"})
                        speakers.add(spk)
                        taken[s] += 1
    for s in need:
        if len(donors[s]) < need[s]:
            print(f"Only {len(donors[s])} of {need[s]} {s} speakers were found in {MSWC_REPO} ({code}_opus); the run goes on with fewer copied voices.")
    detail(f"   {MSWC_REPO}: {len(donors['train'])} train and {len(donors['test'])} test donor speakers from {stats['rows read']} recordings "
           f"in {stats['files']} files, {stats['bytes'] / 1e6:.1f} MB read")
    return donors

def download_librispeech():
    # {set: {speaker: [(flac path, transcript)]}}, and the speakers of the calibration stream and of the test stream (no speaker in both)
    if SMOKE:
        return {}, [], []
    def get(name):
        tgz = Path(WORK_DIR) / f"{name}.tar.gz"
        if not tgz.exists():
            urllib.request.urlretrieve(f"https://www.openslr.org/resources/12/{name}.tar.gz", tgz)
        with tarfile.open(tgz) as t:
            t.extractall(Path(WORK_DIR) / "librispeech", filter="data")
        tgz.unlink()
        out = {}
        for tr in sorted((Path(WORK_DIR) / "librispeech" / "LibriSpeech" / name).glob("*/*/*.trans.txt")):
            for line in tr.read_text().splitlines():
                uid, text = line.split(" ", 1)
                out.setdefault(tr.parent.parent.name, []).append((tr.parent / f"{uid}.flac", text.lower()))
        return out
    with ThreadPoolExecutor(2) as ex:
        libri = dict(zip(("dev-clean", "test-clean"), ex.map(get, ("dev-clean", "test-clean"))))
    detail(f"   LibriSpeech: dev-clean {len(libri['dev-clean'])} speakers, test-clean {len(libri['test-clean'])} speakers")
    tc = sorted(libri["test-clean"])
    random.Random(SEED).shuffle(tc)
    return libri, tc[:20], tc[20:]

In [ ]:
feat_sess = load_featurizer()                 # the listening model that turns sound into numbers
speech = download_spoken_words()              # single words that are not your phrase
sounds = download_sounds()                    # noises and music
donors = find_voices_to_copy()                # real voices to copy in Step 3
rirs = [r[:int(0.6 * SR)] for _, r in load_set(RIR_REPO, 8 if SMOKE else 60, "rir") if len(r) > 100]   # room echoes
libri, cal_speakers, tst_speakers = download_librispeech()   # read-aloud sentences
done("downloads (speech, sounds, LibriSpeech)", f"Step 2 done: downloaded {sum(map(len, speech.values()))} spoken words, "
     f"{sum(map(len, sounds.values()))} sounds and read-aloud sentences by {sum(map(len, libri.values()))} speakers.")

## Step 3 — Say the wake word in many voices

A speech generator, OmniVoice, says your phrase about two thousand times, each time in a different voice. Some clips are kept aside for the test; the ones copied from real speakers use speakers whose voices were not copied for training.

<details>
<summary>More about the voices</summary>

Most voices are invented from a description such as "female, elderly, low pitch, indian accent". About one in eight copies a real speaker from one recorded word. A real speaker is copied for training or for the test, never both, and the cell stops if a voice is on both sides. The spoken-word negatives lost their speaker ids, so a donor speaker may also say one of them. An invented test voice is a new sample of the same kinds of description that made the training voices, so it can sound like one of them; that is why Step 7 counts the copied speakers apart. A clip longer than 1.5 seconds is dropped; if more than a quarter are dropped, the phrase is too long and the run stops. With `DATASET_DIRS` set, the clips are read from those folders. With `SMOKE = True`, a robotic synthesizer (`espeak-ng`) speaks instead.

</details>

In [ ]:
#@title Code for step 3: speaks the phrase (or reads it from DATASET_DIRS) and checks that no voice is on both sides
import csv, subprocess, tempfile

POS_TEXT = WAKE_WORD.strip()
POS_ORIGIN = (f"loaded from {len(DATASETS)} folder(s) written by synthetic_ww_dataset (runs " + ", ".join(m["run_id"] for _, m in DATASETS)
              + "), made with " + "; ".join(sorted({str(m.get("generator", "an unnamed generator")) for _, m in DATASETS}))
              if DATASETS else "generated with espeak-ng (SMOKE)" if SMOKE
              else f"generated with OmniVoice: voice design, and voice cloning from random {MSWC_REPO} speakers")

OTHER_VOICES = ("robotic espeak-ng voice variants not used in training" if SMOKE and not DATASETS
                else "invented voices, new samples of the kinds of description used in training")

def keep(x, sr=24000):
    # the clip at 16 kHz with its silent ends cut off, or None when it does not fit the 1.5 s window
    x = trim(to16k(np.asarray(x, np.float32), sr), db=-35.0)
    return x if 0.2 * SR <= len(x) <= WINDOW else None

def load_voices():
    # the positives of every folder, on the side and with the voice id the folder gives them, without the augmented copies
    pos, skipped = {"train": [], "test": []}, Counter()
    for d, m in DATASETS:
        with open(Path(d) / "metadata.csv", newline="", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                if row["label"] != "1":
                    continue
                if row["source"].endswith("+augmented"):
                    skipped["augmented copies"] += 1
                    continue
                assert row["split"] in pos, f"{d}/metadata.csv: unknown split {row['split']!r}"
                x = load16k(Path(d) / row["path"])
                if not 0.2 * SR <= len(x) <= WINDOW:
                    skipped["longer than 1.5 s or too short"] += 1
                    continue
                pos[row["split"]].append((row["voice_id"], x))
                if sum(map(len, pos.values())) % 500 == 0:
                    detail(f"   read {sum(map(len, pos.values()))} clips")
    detail("   skipped: " + (", ".join(f"{n} {k}" for k, n in skipped.items()) or "nothing"))
    return pos, 0.0

def robot_voices():
    # SMOKE: espeak-ng voice variants, four of them kept for the test
    pos = {"train": [], "test": []}
    variants = [f"en-{a}+{g}{n}" for a in ("us", "gb") for g in "mf" for n in range(1, 5)]
    rng.shuffle(variants)
    for split, n, pool in (("train", N_DESIGN_TRAIN, variants[4:]), ("test", N_DESIGN_TEST, variants[:4])):
        for i in range(n):
            with tempfile.TemporaryDirectory() as d:
                subprocess.run(["espeak-ng", "-v", pool[i % len(pool)], "-p", str(rng.randint(20, 80)), "-s", str(rng.randint(120, 190)),
                                "-w", f"{d}/a.wav", POS_TEXT], check=True, capture_output=True)
                x = keep(*sf.read(f"{d}/a.wav", dtype="float32"))
            if x is not None:
                pos[split].append((f"espeak:{pool[i % len(pool)]}", x))
    return pos, 0.0

def generate_voices():
    # OmniVoice: invented voices (a random description and speed), then voices copied from the donors of Step 2
    from omnivoice import OmniVoice
    tts = OmniVoice.from_pretrained("k2-fsa/OmniVoice", device_map="cuda:0", dtype=torch.float16)
    pos, dropped, seconds = {"train": [], "test": []}, 0, 0.0

    def describe(r):
        parts = [r.choice(["male", "female"]), r.choice(["child", "teenager", "young adult", "middle-aged", "elderly"]),
                 r.choice(["very low pitch", "low pitch", "moderate pitch", "high pitch", "very high pitch"])]
        if LANGUAGE.lower() == "english":
            parts.append(r.choice(["american accent", "british accent", "australian accent", "canadian accent", "indian accent"]))
        return ", ".join(parts)

    def generate(**kw):
        # A batch with one degenerate item (an empty waveform) is retried item by item; failed items are dropped.
        try:
            return tts.generate(**kw)
        except ValueError:
            out = []
            for i in range(len(kw["text"])):
                try:
                    out.append(tts.generate(**{k: ([v[i]] if isinstance(v, list) else v) for k, v in kw.items()})[0])
                except ValueError:
                    out.append(None)
            return out

    def batches():
        # (side, voice ids, what OmniVoice is told) for every batch
        for split, n in (("train", N_DESIGN_TRAIN), ("test", N_DESIGN_TEST)):
            r = random.Random(f"{SEED}:{split}:design")
            for start in range(0, n, 32):
                m = min(32, n - start)
                yield split, [f"design:{split}:{start + j}" for j in range(m)], dict(
                    instruct=[describe(r) for _ in range(m)], speed=[round(r.uniform(1.0, 1.4), 2) for _ in range(m)])
        for split in ("train", "test"):
            r = random.Random(f"{SEED}:{split}:clone")
            for start in range(0, len(donors[split]), 12):
                batch = donors[split][start:start + 12]
                # the word a donor says is the transcript OmniVoice needs for the reference
                yield split, [f"mswc:{d['speaker'][:12]}" for d in batch], dict(
                    ref_audio=[(torch.from_numpy(d["audio"]).unsqueeze(0), SR) for d in batch],
                    ref_text=[d["keyword"] for d in batch], speed=[round(r.uniform(1.0, 1.3), 2) for _ in batch])

    for split, ids, told in batches():
        t0 = time.time()
        out = generate(text=[POS_TEXT] * len(ids), language=LANGUAGE, **told)
        seconds += time.time() - t0
        detail(f"   {split}: {len(pos[split])} clips so far")
        for voice, a in zip(ids, out):
            x = None if a is None else keep(a)
            if x is None:
                dropped += 1
            else:
                pos[split].append((voice, x))
    total = sum(len(v) for v in pos.values()) + dropped
    detail(f"   generated {total} clips in {seconds:.0f} s ({total / seconds:.2f} clips/s); dropped {dropped} longer than 1.5 s or too short")
    assert dropped <= 0.25 * total, f"{dropped} of {total} clips were longer than the 1.5 s window; the phrase {POS_TEXT!r} is too long for it."
    del tts
    torch.cuda.empty_cache()
    return pos, seconds

def check_voices(pos):
    # the distinct voices of each side; stops if a voice or a donor speaker is on both sides
    voices = {s: {v for v, _ in pos[s]} for s in pos}
    assert not voices["train"] & voices["test"], "a voice is on both sides of the split"
    donor_spk = {s: {d["speaker"] for d in donors[s]} for s in donors}
    assert all(mswc_side(spk) == s for s in donor_spk for spk in donor_spk[s]), "a donor speaker is on the wrong side"
    assert not donor_spk["train"] & donor_spk["test"], "a donor speaker is on both sides"
    detail(f"   copied voices: {sum(v.startswith('mswc:') for v, _ in pos['train'])} train and {sum(v.startswith('mswc:') for v, _ in pos['test'])} test clips, "
           f"{len(donor_spk['train'])} and {len(donor_spk['test'])} donor speakers, none on both sides")
    assert len(pos["train"]) >= (8 if SMOKE or DATASETS else 800) and len(pos["test"]) >= 4, "too few clips of the phrase to train on"
    if DATASETS and len(pos["train"]) < 800:
        print(f"Only {len(pos['train'])} training clips: one full run of synthetic_ww_dataset gives about 2,000.")
    return voices

In [ ]:
pos, GEN_SECONDS = load_voices() if DATASETS else robot_voices() if SMOKE else generate_voices()
voices = check_voices(pos)                    # no voice may be in both the training and the test
done("load positives" if DATASETS else "generate positives", f"Step 3 done: {POS_TEXT!r} is spoken in {len(pos['train'])} training clips "
     f"({len(voices['train'])} voices) and {len(pos['test'])} test clips ({len(voices['test'])} other voices).")

## Step 4 — Prepare the training examples

Every sound is cut into pieces of 1.5 seconds, and the training pieces get room echo and background noise. Three long recordings without your phrase are also put together: one to practise on, one to set the sensitivity, one for the final test.

<details>
<summary>More about the three recordings</summary>

The practice recording (the training stream) is about four hours; the other two (the calibration stream and the test stream) are two hours each, 40 % sound and music. They share no clip, no word and no speaker. Silence and faint hiss are added to the training pieces, so a quiet microphone does not wake the assistant.

</details>

In [ ]:
#@title Code for step 4: cuts 1.5 s windows, adds echo and noise, builds the three streams
def peak_to(x, level):
    p = np.abs(x).max()
    return x * (level / p) if p > 0 else x

def fit_window(x, r):
    # a longer clip is cropped at a random position; a shorter one is placed at a random position in silence
    if len(x) >= WINDOW:
        s = int(r.integers(0, len(x) - WINDOW + 1))
        return x[s:s + WINDOW].copy()
    w = np.zeros(WINDOW, np.float32)
    s = int(r.integers(0, WINDOW - len(x) + 1))
    w[s:s + len(x)] = x
    return w

def augment(w, r, noises):
    if rirs and r.random() < 0.3:                      # reverb: convolve with a recorded room response
        peak = np.abs(w).max()
        w = fftconvolve(w, rirs[int(r.integers(len(rirs)))])[:WINDOW]
        w = w * (peak / (np.abs(w).max() + 1e-8))
    w = w * 10 ** (r.uniform(-9, 3) / 20)
    if noises and r.random() < 0.8:                    # background sound, 0 to 25 dB below the signal
        bg = fit_window(noises[int(r.integers(len(noises)))], r)
        bg_rms, w_rms = np.sqrt((bg ** 2).mean()) + 1e-8, np.sqrt((w ** 2).mean()) + 1e-8
        w = w + bg * (w_rms / bg_rms) * 10 ** (-r.uniform(0, 25) / 20)
    return np.clip(w, -1, 1).astype(np.float32)

def silence_windows(n, r):
    # digital zeros, and white noise at -60 and -40 dB
    return [np.zeros(WINDOW, np.float32) if level is None else np.clip(r.standard_normal(WINDOW) * 10 ** (level / 20), -1, 1).astype(np.float32)
            for level in ([None, -60.0, -40.0][i % 3] for i in range(n))]

def training_set(r):
    # windows, labels (1 is the phrase) and a group number 0 to 9; group 0 is the validation part
    noise = [x for _, x in sounds["train"]]
    sentences = []                                     # read-aloud sentences, cut at random
    if N_SENTENCE_WINDOWS:
        utts = [u for spk in libri["dev-clean"].values() for u in spk]
        sentences = [load16k(path) for path, _ in random.Random(SEED).sample(utts, min(N_SENTENCE_WINDOWS, len(utts)))]
    pools = [(1, [peak_to(x, 0.7) for _, x in pos["train"]], 3),      # (label, clips, windows made from each clip)
             (0, [peak_to(x, 0.7) for _, x in speech["train"]], 2),
             (0, [peak_to(x, float(r.uniform(0.02, 0.8))) for x in noise], "auto"),
             (0, [peak_to(x, float(r.uniform(0.1, 0.8))) for x in sentences], 1)]
    wins, labels, groups = [], [], []
    for label, clips, times in pools:
        for gi, clip in enumerate(clips):
            for _ in range((2 if len(clip) > 5 * SR else 1) if times == "auto" else times):   # long clips give two crops
                wins.append(augment(fit_window(clip, r), r, noise))
                labels.append(label)
                groups.append(gi % 10)
    for w in silence_windows(N_SILENCE_TRAIN, r):
        wins.append(w); labels.append(0); groups.append(int(r.integers(10)))
    return np.stack(wins), np.array(labels, np.float32), np.array(groups)

def test_set(r):
    # clean windows of the kept-aside clips, by kind
    return {"wake phrase": np.stack([fit_window(peak_to(x, 0.7), r) for _, x in pos["test"]]),
            "speech": np.stack([fit_window(peak_to(x, 0.7), r) for _, x in speech["test"]]),
            "noise": np.stack([fit_window(peak_to(x, float(r.uniform(0.02, 0.8))), r) for _, x in sounds["test"][:max(N_SOUND_TEST, 1)]]),
            "silence": np.stack(silence_windows(N_SILENCE_TEST, r))}

def sentence_parts(speakers, split, seconds, seed):
    utts = [(spk, u) for spk in speakers for u in libri[split][spk]]
    random.Random(seed).shuffle(utts)
    parts, total = [], 0.0
    for spk, (path, text) in utts:
        if total >= seconds:
            break
        x = load16k(path)
        total += len(x) / SR
        parts.append(("LibriSpeech sentence", f"{spk}/{path.stem}: {text}", x))
    return parts

def sound_parts(named, seconds, seed):
    # a random subset of sound clips adding up to `seconds` (all of them when seconds is 0)
    named = named[:]
    random.Random(seed).shuffle(named)
    out, total = [], 0.0
    for n, x in named:
        if seconds and total >= seconds:
            break
        out.append(("sound: " + n.split("/")[0], n, x)); total += len(x) / SR
    return out

def build_stream(parts, seed):
    # one long recording, and for every segment (start, end, kind, name) so that a false wake-up can be traced to its source
    r = np.random.default_rng(seed)
    gap = np.zeros(int(0.3 * SR), np.float32)
    audio, index, t = [], [], 0.0
    for i in r.permutation(len(parts)):
        src, name, x = parts[i]
        x = peak_to(x, float(r.uniform(0.05, 0.8) if src.startswith("sound") else r.uniform(0.2, 0.8)))
        index.append((t, t + len(x) / SR, src, name))
        audio += [x, gap]
        t += len(x) / SR + 0.3
    return np.concatenate(audio).astype(np.float32), index

def build_streams():
    # the training, calibration and test streams; they share no clip, word or speaker and none holds the phrase
    words = lambda named: [("speech word", n, x) for n, x in named]
    duration = lambda parts: sum(len(x) for *_, x in parts) / SR
    train = words(speech["train"]) + sound_parts(sounds["train"], MINE_SOUND_SECONDS, SEED + 10)
    if MINE_SPEECH_SECONDS:
        train += sentence_parts(sorted(libri["dev-clean"]), "dev-clean", MINE_SPEECH_SECONDS - duration(words(speech["train"])), SEED + 11)
    cal = words(speech["cal"]) + sound_parts(sounds["cal"], SOUND_SHARE * STREAM_SECONDS, SEED + 17)
    tst = words(speech["tst"]) + sound_parts(sounds["tst"], SOUND_SHARE * STREAM_SECONDS, SEED + 18)
    if STREAM_SECONDS:
        cal += sentence_parts(cal_speakers, "test-clean", STREAM_SECONDS - duration(cal), SEED + 13)
        tst += sentence_parts(tst_speakers, "test-clean", STREAM_SECONDS - duration(tst), SEED + 14)
    streams = [build_stream(train, SEED + 12), build_stream(cal, SEED + 15), build_stream(tst, SEED + 16)]
    for name, (a, ix) in zip(("training", "calibration", "test"), streams):
        kinds = Counter()
        for t0, t1, src, _ in ix:
            kinds[src.split(":")[0]] += t1 - t0
        detail(f"   {name:11s} stream: {len(a) / SR / 3600:.2f} h, {len(ix)} segments, {100 * kinds['sound'] / max(1, sum(kinds.values())):.0f} % sound and music ("
               + ", ".join(f"{k} {v / 60:.0f} min" for k, v in kinds.items()) + ")")
    return streams

In [ ]:
train_audio, train_labels, train_groups = training_set(np.random.default_rng(SEED))   # pieces to learn from
test_audio = test_set(np.random.default_rng(SEED + 1))                                # pieces kept for the test
detail(f"   {len(train_audio)} training windows cut")
train_feats = featurize(train_audio)
detail("   training windows featurized")
tick("windows and features")
(mine_audio, mine_index), (cal_audio, cal_index), (tst_audio, tst_index) = build_streams()
CAL_HOURS, STREAM_HOURS = len(cal_audio) / SR / 3600, len(tst_audio) / SR / 3600
done("streams", f"Step 4 done: {len(train_audio)} training pieces ({int(train_labels.sum())} with the phrase), {sum(map(len, test_audio.values()))} test pieces, "
     f"and recordings of {len(mine_audio) / SR / 3600:.2f}, {CAL_HOURS:.2f} and {STREAM_HOURS:.2f} hours.")

## Step 5 — Train the model

The model learns to tell your phrase from everything else. Then it listens to the practice recording and is trained again on whatever fooled it, up to five times.

<details>
<summary>More about the training</summary>

The model is one small recurrent layer (a GRU with 128 units) on top of the frozen listening model. A tenth of the pieces is kept for validation, and the best epoch is kept. Each round scores the practice recording every 80 ms, as the plugin does with live audio, and adds the moments that score above 0.1 as new "do not wake" examples. The rounds stop when fewer than five moments still score 0.5 or more.

</details>

In [ ]:
#@title Code for step 5: the model, its training loop, and the search for sounds that fool it
import torch.nn as nn

class Head(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(FEATURE_DIM, 128, batch_first=True)
        self.fc = nn.Linear(128, 1)
    def forward(self, features):                 # [batch, frames, 128] -> [batch, 1] logit
        out, _ = self.gru(features)
        return self.fc(out[:, -1])

def train_head(feats, labels, groups, tag):
    # trains a new head and returns it at the epoch with the lowest validation loss, with that loss
    torch.manual_seed(SEED)
    head = Head().to(DEVICE)
    opt = torch.optim.Adam(head.parameters(), lr=LR)
    loss_fn = nn.BCEWithLogitsLoss()
    X = torch.from_numpy(feats).to(DEVICE)
    Y = torch.from_numpy(labels).to(DEVICE).unsqueeze(1)
    is_val = torch.from_numpy(groups == 0).to(DEVICE)
    if is_val.sum() == 0 or is_val.all():
        is_val = torch.arange(len(X), device=DEVICE) % 10 == 0
    Xtr, Ytr, Xva, Yva = X[~is_val], Y[~is_val], X[is_val], Y[is_val]
    best = (float("inf"), None)
    for epoch in range(1, EPOCHS + 1):
        head.train()
        perm = torch.randperm(len(Xtr), device=DEVICE)
        for i in range(0, len(perm), BATCH_SIZE):
            idx = perm[i:i + BATCH_SIZE]
            opt.zero_grad()
            loss_fn(head(Xtr[idx]), Ytr[idx]).backward()
            opt.step()
        head.eval()
        with torch.no_grad():
            val_logits = head(Xva)
            val_loss = loss_fn(val_logits, Yva).item()
        if val_loss < best[0]:
            best = (val_loss, {k: v.detach().cpu().clone() for k, v in head.state_dict().items()})
        if epoch == 1 or epoch % 10 == 0 or epoch == EPOCHS:
            detail(f"   [{tag}] epoch {epoch:3d}/{EPOCHS}  validation loss {val_loss:.4f}  accuracy {((val_logits > 0).float() == Yva).float().mean().item():.3f}")
    head.load_state_dict(best[1])
    del X, Y, Xtr, Xva
    return head, best[0]

def score_feats(head, feats16, bs=2048):
    head.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(feats16), bs):
            x = torch.from_numpy(feats16[i:i + bs].astype(np.float32)).to(DEVICE)
            out.append(torch.sigmoid(head(x)).squeeze(1).cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0, np.float32)

def stream_features(audio):
    # the features of every 1.5 s window of a stream, one every 80 ms
    starts = np.arange(0, len(audio) - WINDOW + 1, HOP)
    feats = np.empty((len(starts), N_FRAMES, FEATURE_DIM), np.float16)
    for i in range(0, len(starts), 64):
        b = np.stack([audio[s:s + WINDOW] for s in starts[i:i + 64]])
        feats[i:i + len(b)] = featurize(b)
        if i and i % 20480 == 0:
            detail(f"   featurized {i} of {len(starts)} windows of the training stream")
    return feats

def pick_hard(scores, floor, limit, min_gap=3):
    # the highest scoring windows, never two within 0.24 s of each other
    chosen, taken = [], np.zeros(len(scores), bool)
    for i in np.argsort(-scores):
        if scores[i] < floor or len(chosen) >= limit:
            break
        if taken[max(0, i - min_gap): i + min_gap + 1].any():
            continue
        taken[i] = True
        chosen.append(int(i))
    return chosen

def train_and_mine():
    # trains, then adds the training-stream windows the head wrongly likes and trains again, until few are left
    t0 = time.time()
    head, loss = train_head(train_feats, train_labels, train_groups, "round 0")
    detail("   round 0 trained")
    mine_feats = stream_features(mine_audio)
    pos_f, base_neg = train_feats[train_labels == 1], train_feats[train_labels == 0]
    mr = np.random.default_rng(SEED + 21)
    rand_idx = mr.choice(len(mine_feats), size=min(MAX_RANDOM_STREAM, len(mine_feats)), replace=False)
    hard_idx = set()
    for rnd in range(1, MAX_MINING_ROUNDS + 1):
        scores = score_feats(head, mine_feats)
        n_half = int((scores >= 0.5).sum())
        detail(f"   [round {rnd}] training stream: {n_half} of {len(scores)} windows score at least 0.5, {int((scores >= MINE_FROM).sum())} at least {MINE_FROM}")
        if rnd > 1 and n_half < STOP_BELOW:
            break
        chosen = pick_hard(scores, MINE_FROM, MAX_HARD_PER_ROUND)
        hard_idx |= set(chosen)
        if len(hard_idx) > MAX_HARD_TOTAL:
            hard_idx = set(mr.choice(sorted(hard_idx), size=MAX_HARD_TOTAL, replace=False).tolist())
        detail(f"   [round {rnd}] mined {len(chosen)} windows; {len(hard_idx)} hard negatives in total")
        feats = np.concatenate([pos_f, base_neg, mine_feats[rand_idx].astype(np.float32), mine_feats[sorted(hard_idx)].astype(np.float32)])
        labels = np.concatenate([np.ones(len(pos_f)), np.zeros(len(feats) - len(pos_f))]).astype(np.float32)
        head, loss = train_head(feats, labels, mr.integers(0, 10, len(feats)), f"round {rnd}")
        del feats
    scores = score_feats(head, mine_feats)
    detail(f"   kept the epoch with validation loss {loss:.4f}")
    return head, int((scores >= 0.5).sum()), len(scores), time.time() - t0

In [ ]:
head, still_fooled, n_moments, TRAIN_SECONDS = train_and_mine()
done("train and mine", f"Step 5 done: the model is trained. {still_fooled} of {n_moments} moments of the practice recording still fool it.")

## Step 6 — Save the model and set its sensitivity

Writes the model file `best_f1.onnx`. Then it picks the threshold, the score from 0 to 1 the model must reach before it wakes: the lowest one that gives at most 2 false wake-ups per hour on the second recording.

<details>
<summary>More about the threshold</summary>

The OVOS plugin scores the calibration stream, which holds no wake phrase, so every wake-up on it is false. The threshold is stored inside the model file and in `ovos_config.json`. It is a choice made on one two-hour recording. It is not the longer stream calibration of the published [`OpenVoiceOS/wakehubert-wakewords`](https://huggingface.co/OpenVoiceOS/wakehubert-wakewords) models.

</details>

In [ ]:
#@title Code for step 6: writes the model file in the plugin's format and chooses the threshold
import onnx
from scipy.stats import chi2
from ovos_utils.log import LOG
from ovos_ww_plugin_wakeforge import WakeForgeHotwordPlugin

if not SHOW_DETAILS:
    LOG.set_level("WARNING")                            # the plugin logs a line every time it loads the model

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
HEAD_PATH = Path(OUTPUT_DIR) / "best_f1.onnx"

def export_head(head_cpu, threshold=None):
    # the ONNX file, with the metadata the plugin reads (featurizer, window length and, once chosen, the threshold)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")                 # torch warns that this exporter is the older one
        torch.onnx.export(head_cpu, torch.from_numpy(train_feats[:2]), str(HEAD_PATH), input_names=["features"], output_names=["logits"],
                          dynamic_axes={"features": {0: "batch", 1: "frames"}, "logits": {0: "batch"}}, opset_version=17, dynamo=False)
    model = onnx.load(str(HEAD_PATH))
    meta = {
        "pretrained_featurizer": FEATURIZER_NAME, "featurizer_revision": FEATURIZER_REVISION,
        "window_frames": N_FRAMES, "feature_dim": FEATURE_DIM, "frame_rate_hz": 50.0, "arch": "gru",
        "wake_word": WAKE_WORD, "license": "Apache-2.0",
        "training_data": f"positives {POS_ORIGIN}; negatives from {COLLECTION} and LibriSpeech dev-clean; room responses from {RIR_REPO}; generated silence",
    }
    if threshold is not None:
        meta["default_threshold"] = repr(float(threshold))
        meta["threshold_note"] = f"chosen on a held-out stream of {CAL_HOURS:.2f} h for at most {CAL_MAX_FA_PER_HOUR} false wake-ups per hour; not stream-calibrated"
    for key, value in meta.items():
        entry = model.metadata_props.add()
        entry.key, entry.value = key, str(value)
    onnx.save(model, str(HEAD_PATH))

def save_model(head):
    # writes the file without a threshold and checks that onnxruntime gives the same scores as torch
    head_cpu = Head()
    head_cpu.load_state_dict({k: v.cpu() for k, v in head.state_dict().items()})
    head_cpu.eval()
    export_head(head_cpu)
    check = ort.InferenceSession(str(HEAD_PATH), opts, providers=["CPUExecutionProvider"])
    for n in (75, 40):                                  # the plugin may score other frame counts
        f = train_feats[:8, :n]
        with torch.no_grad():
            want = head_cpu(torch.from_numpy(f)).numpy().reshape(-1)
        assert np.abs(check.run(None, {"features": f})[0].reshape(-1) - want).max() < 1e-3, f"onnxruntime and torch disagree at {n} frames"
    return head_cpu

def load_plugin(threshold):
    # the class OVOS creates from the hotwords section of mycroft.conf, loading the model by file path
    return WakeForgeHotwordPlugin(key_phrase=WORD_SLUG, config={"module": "ovos-ww-plugin-wakeforge", "model": str(HEAD_PATH),
                                                                "onnx_threads": THREADS, "threshold": threshold})

def to_int16(a):
    return (np.clip(a, -1, 1) * 32767).astype(np.int16)

def stream_scores(audio):
    # the plugin's raw score for every 80 ms block of an unbroken feed (a threshold of 2 never fires)
    pcm, pl, scores = to_int16(audio), load_plugin(2.0), []
    for i in range(0, len(pcm) - HOP + 1, HOP):
        pl.update(pcm[i:i + HOP].tobytes())
        scores.append(pl.last_score or 0.0)
        if len(scores) % 9000 == 0:
            detail(f"   calibration stream: scored {i / SR / 60:.0f} min")
    return np.array(scores)

def simulate(scores, threshold):
    # what a head that states default_threshold does: fire on one score >= threshold, then ignore 2 s
    fires, quiet = [], 0
    for k, sc in enumerate(scores):
        if quiet > 0:
            quiet -= HOP
        elif sc >= threshold:
            fires.append((k + 1) * HOP / SR)
            quiet = 2 * SR
    return fires

def poisson_ci(k, hours):
    # exact 95 % interval for a rate, from k events in `hours` hours
    return 0.0 if k == 0 else chi2.ppf(0.025, 2 * k) / 2 / hours, chi2.ppf(0.975, 2 * k + 2) / 2 / hours

def by_source(fires, index, title, top=8):
    # SHOW_DETAILS: which kinds of segment, and which segments, the wake-ups fall on
    starts = np.array([t0 for t0, *_ in index])
    seconds, count, offenders = Counter(), Counter(), Counter()
    for t0, t1, src, _ in index:
        seconds[src] += t1 - t0
    for t in fires:
        k = max(0, int(np.searchsorted(starts, t, side="right")) - 1)
        count[index[k][2]] += 1
        offenders[index[k][2:]] += 1
    detail(f"   {title}: {len(fires)} wake-ups")
    for src in sorted(seconds):
        detail(f"      {src:<42s} {count[src]:>4d} wake-ups in {seconds[src] / 3600:.2f} h")
    for (_, name), n in offenders.most_common(top):
        detail(f"      {n}x  {name[:110]}")

def clip_peak(pl, window):
    # the highest score the plugin gives one clip
    pl.reset()
    pcm, peak = to_int16(np.concatenate([window, np.zeros(int(0.4 * SR), np.float32)])), 0.0
    for i in range(0, len(pcm) - HOP + 1, HOP):
        pl.update(pcm[i:i + HOP].tobytes())
        peak = max(peak, pl.last_score or 0.0)
    return peak

def choose_threshold():
    # the lowest candidate with at most CAL_MAX_FA_PER_HOUR false wake-ups per hour on the calibration stream, and that rate
    cal_scores = stream_scores(cal_audio)
    by_source(simulate(cal_scores, 0.5), cal_index, "calibration stream at threshold 0.5")
    pl = load_plugin(2.0)
    peaks = np.array([clip_peak(pl, w) for w in test_audio["wake phrase"]])   # shown beside each candidate, not used to choose
    detail(f"   scored {len(peaks)} test clips of the phrase")
    detail(f"   {'threshold':>10} {'wake-ups':>9} {'per hour':>9}   {'recall on held-out synthetic voices':>36}")
    threshold = None
    for cand in CANDIDATES:
        n = len(simulate(cal_scores, cand))
        detail(f"   {cand:>10} {n:>9} {n / CAL_HOURS:>9.2f}   {int((peaks >= cand).sum()):>5} of {len(peaks)} = {(peaks >= cand).mean():>6.1%}")
        if threshold is None and n / CAL_HOURS <= CAL_MAX_FA_PER_HOUR:
            threshold = cand
    if threshold is None:
        threshold = CANDIDATES[-1]
        print(f"No threshold gives {CAL_MAX_FA_PER_HOUR} false wake-ups per hour or fewer on the calibration stream; using the highest, {threshold}.")
    n = len(simulate(cal_scores, threshold))
    detail("   95 % interval of the rate at the chosen threshold: {:.2f} to {:.2f} per hour".format(*poisson_ci(n, CAL_HOURS)))
    return threshold, n / CAL_HOURS

def write_ovos_config(threshold):
    cfg = {
        "listener": {"wake_word": WORD_SLUG},
        "hotwords": {WORD_SLUG: {
            "module": "ovos-ww-plugin-wakeforge",
            "model": "~/.local/share/mycroft/best_f1.onnx",     # where you copied the file to
            "threshold": threshold,                              # chosen on a held-out stream; also stored in the file
        }},
    }
    (Path(OUTPUT_DIR) / "ovos_config.json").write_text(json.dumps(cfg, indent=2) + "\n")
    return cfg

In [ ]:
head_cpu = save_model(head)                    # writes best_f1.onnx and checks the file
THRESHOLD, CAL_FA = choose_threshold()
export_head(head_cpu, THRESHOLD)               # writes the file again, with the threshold stored inside
cfg = write_ovos_config(THRESHOLD)             # the settings for OpenVoiceOS
done("threshold (calibration stream)", f"Step 6 done: saved {HEAD_PATH} with threshold {THRESHOLD} "
     f"({CAL_FA:.2f} false wake-ups per hour on the {CAL_HOURS:.2f} hour calibration recording).")

## Step 7 — Test the model

The model is loaded the way OpenVoiceOS loads it and tested twice. First on the clips kept aside, counting real speakers whose voices were not copied for training apart from the other voices, then on a recording of speech, sound and music it has never heard, where every wake-up is a mistake.

In [ ]:
#@title Code for step 7: feeds the test clips and the test stream to the OVOS plugin
def run_clip(plugin, window):
    # After a wake-up the plugin ignores audio for two seconds, and reset() does not end that pause. Two seconds of
    # silence before each clip let the pause run out, so one clip's wake-up cannot hide the next clip's.
    plugin.reset()
    for _ in range(28):
        plugin.update(np.zeros(HOP, np.int16).tobytes())
    plugin.found_wake_word()
    plugin.reset()
    pcm, fired = to_int16(np.concatenate([window, np.zeros(int(0.4 * SR), np.float32)])), False
    for i in range(0, len(pcm), HOP):                  # 80 ms blocks
        plugin.update(pcm[i:i + HOP].tobytes())
        fired = plugin.found_wake_word() or fired
    return fired

def smoke_check():
    # the saved file, loaded with the saved settings, must state the chosen threshold, score as torch does,
    # wake on a clear majority of the test clips of the phrase and rank them above the other test clips
    saved = {p.key: p.value for p in onnx.load(str(HEAD_PATH)).metadata_props}
    threshold = json.loads((Path(OUTPUT_DIR) / "ovos_config.json").read_text())["hotwords"][WORD_SLUG]["threshold"]
    assert "default_threshold" in saved and float(saved["default_threshold"]) == threshold == THRESHOLD, (
        f"the saved file states threshold {saved.get('default_threshold')}, the saved settings {threshold}, the run chose {THRESHOLD}")
    assert saved.get("pretrained_featurizer") == FEATURIZER_NAME, "the saved file does not name its featurizer"
    pl = load_plugin(threshold)
    def aligned(w):                                    # the plugin's score once its 1.5 s buffer holds exactly this window
        pl.reset()
        pcm = to_int16(np.concatenate([np.zeros(-WINDOW % HOP, np.float32), w]))
        for i in range(0, len(pcm), HOP):
            pl.update(pcm[i:i + HOP].tobytes())
        return pl.last_score
    wins = np.concatenate(list(test_audio.values()))
    is_phrase = np.arange(len(wins)) < len(test_audio["wake phrase"])
    got = np.array([aligned(w) for w in wins])
    want = score_feats(head, featurize(to_int16(wins).astype(np.float32) / 32768))
    assert np.abs(got - want).max() < 0.02, f"the plugin and torch disagree by up to {np.abs(got - want).max():.3f} on the same windows"
    hits = sum(run_clip(load_plugin(threshold), w) for w in test_audio["wake phrase"])
    n = len(test_audio["wake phrase"])
    assert 3 * hits >= 2 * n, f"the saved model woke on only {hits} of {n} test clips of the phrase at its threshold {threshold}"
    p, q = got[is_phrase], got[~is_phrase]
    auc = (p[:, None] > q[None]).mean() + 0.5 * (p[:, None] == q[None]).mean()
    assert auc > 0.9, f"the plugin ranks the phrase above other sound in only {auc:.1%} of pairs"
    print(f"SMOKE check passed: the saved model woke on {hits} of {n} phrase clips at threshold {threshold}; "
          f"plugin and torch agree within {np.abs(got - want).max():.4f}; phrase ranked above other sound in {auc:.1%} of pairs")

def test_clips(plugin):
    # {kind: (clips that woke the plugin, clips)}; the phrase is counted apart for copied real speakers and for the other voices
    fired = {kind: [run_clip(plugin, w) for w in wins] for kind, wins in test_audio.items()}
    copied = [v.startswith("mswc:") for v, _ in pos["test"]]
    counts = {kind: (sum(f), len(f)) for kind, f in fired.items()}
    for kind, keep in (("copied", True), ("other", False)):
        f = [hit for hit, c in zip(fired["wake phrase"], copied) if c == keep]
        counts[kind] = (sum(f), len(f))
    detail("   woke the plugin: " + ", ".join(f"{kind} {hit} of {n}" for kind, (hit, n) in counts.items()))
    return counts

def test_stream():
    # the times of the wake-ups on the test stream, fed in 80 ms blocks with no reset; every one is false
    pcm, pl, fires, scores = to_int16(tst_audio), load_plugin(THRESHOLD), [], []
    for i in range(0, len(pcm) - HOP + 1, HOP):
        pl.update(pcm[i:i + HOP].tobytes())
        scores.append(pl.last_score or 0.0)
        if pl.found_wake_word():
            fires.append((i + HOP) / SR)
        if len(scores) % 9000 == 0:
            detail(f"   test stream: streamed {i / SR / 60:.0f} min, {len(fires)} false wake-ups so far")
    detail(f"   replaying the recorded scores with the same rule gives {len(simulate(np.array(scores), THRESHOLD))} wake-ups")
    by_source(fires, tst_index, "test stream")
    return fires

In [ ]:
plugin = load_plugin(THRESHOLD)                # the same plugin OpenVoiceOS uses
assert plugin.threshold == THRESHOLD
counts = test_clips(plugin)
synth_hit, synth_n = counts["wake phrase"]
HEARD = ([(counts["copied"], f"clips copied from real speakers whose voices were not copied for training")] if counts["copied"][1] else []) + [
    (counts["other"], f"clips of {OTHER_VOICES}"), ((synth_hit, synth_n), "test clips in all")]
synth_false, synth_n_neg = (sum(counts[k][i] for k in ("speech", "noise", "silence")) for i in (0, 1))
tick("synthetic held-out test")
fires = test_stream()
FA_PER_HOUR, (FA_LO, FA_HI) = len(fires) / STREAM_HOURS, poisson_ci(len(fires), STREAM_HOURS)
done("test stream", "Step 7 done: the model heard the phrase in " + "; ".join(f"{h} of {n} {label}" for (h, n), label in HEARD)
     + f". False wake-ups in {STREAM_HOURS:.2f} hours of other sound: {len(fires)}.")

## What you got

**The model file** is `wakehubert_output/best_f1.onnx`. On Kaggle it is under the Output tab. On Colab it is in the file panel and disappears when the session ends, so download it first.

**To use it in OpenVoiceOS**, copy `best_f1.onnx` to `~/.local/share/mycroft/` on your device and run `pip install --pre ovos-ww-plugin-wakeforge` there. Then add the settings printed below to `~/.config/mycroft/mycroft.conf`. They are also saved as `wakehubert_output/ovos_config.json`.

**The numbers** are printed below. When the summary has a line for clips copied from real speakers whose voices were not copied for training, that line is the fairest test; otherwise read the line for all test clips. A higher share of "heard the phrase" is better; fewer false wake-ups per hour is better. They describe this one run, with computer-made voices and public recordings, so they do not say how the model behaves in your home.

<details>
<summary>Read more</summary>

- [Wake word plugins](https://openvoiceos.github.io/ovos-technical-manual/312-wake_word_plugins/) in the OVOS manual
- [ovos-ww-plugin-wakeforge](https://github.com/OpenVoiceOS/ovos-ww-plugin-wakeforge), the plugin that loads the model
- [WakeHuBERT](https://huggingface.co/TigreGotico/wakehubert-tiny), the listening model, and [OmniVoice](https://github.com/k2-fsa/OmniVoice), the speech generator
- [wakeforge](https://github.com/TigreGotico/wakeforge), the full training toolkit; this notebook does not use it
- [`synthetic_ww_dataset.ipynb`](synthetic_ww_dataset.ipynb), to make the voices beforehand, in bulk

</details>

In [ ]:
print(f"""Your wake word model for {WAKE_WORD!r}: {HEAD_PATH}
Settings for OpenVoiceOS      : {Path(OUTPUT_DIR) / 'ovos_config.json'}

Heard the phrase   : {(chr(10) + ' ' * 21).join(f'{h} of {n} {label}' + (f' ({h / n:.1%})' if n else '') for (h, n), label in HEARD)}
Woke by mistake    : {synth_false} of {synth_n_neg} short clips of other speech, noise and silence
False wake-ups     : {len(fires)} in {STREAM_HOURS:.2f} h of speech, sound and music = {FA_PER_HOUR:.2f} per hour (95 % interval {FA_LO:.2f} to {FA_HI:.2f})
Threshold          : {THRESHOLD}, chosen on a separate {CAL_HOURS:.2f} h recording ({CAL_FA:.2f} false wake-ups per hour on it)
Clips of the phrase: {len(pos['train'])} for training, {len(pos['test'])} for the test; {POS_ORIGIN}{'' if DATASETS else f' in {GEN_SECONDS:.0f} s'}
Voices             : {len(voices['train'])} in training, {len(voices['test'])} in the test, no voice id in both
Run                : {'SMOKE' if SMOKE else 'full'}, device {DEVICE}, {EPOCHS} epochs, training {TRAIN_SECONDS:.0f} s
Minutes per step   : {', '.join(f'{k} {v / 60:.0f}' for k, v in _STEPS.items())} (total {sum(_STEPS.values()) / 60:.0f})
These numbers describe this run only; they do not say how the model behaves in a real home.

Add this to ~/.config/mycroft/mycroft.conf on your device:
{json.dumps(cfg, indent=2)}""")

if SMOKE:   # the trial is too small to measure the model; it checks the saved file the way OpenVoiceOS will use it
    smoke_check()

[![NGI0 Commons Fund](../ngi.png)](https://nlnet.nl/project/OpenVoiceOS)

This project was funded through the [NGI0 Commons Fund](https://nlnet.nl/commonsfund), a fund established by [NLnet](https://nlnet.nl) with financial support from the European Commission's [Next Generation Internet](https://ngi.eu) programme, under the aegis of DG Communications Networks, Content and Technology under grant agreement No [101135429](https://cordis.europa.eu/project/id/101135429).

> This notebook was written with AI assistance (Claude, via Claude Code). Verify outputs before relying on them.

Licensed under Apache-2.0.